In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install torch torchvision torchaudio

In [ ]:
import torch
print(torch.cuda.is_available())  # Should return True
print(torch.cuda.get_device_name(0))  # Should return 'Tesla T4'


In [ ]:
!pip install bitsandbytes

In [ ]:
import bitsandbytes as bnb
print(bnb.__version__)  # Check the installed version


In [ ]:
!pip install -qqq -U git+https://github.com/huggingface/transformers.git@e03a9cc
!pip install -qqq -U git+https://github.com/huggingface/peft.git@42a184f
!pip install -qqq -U git+https://github.com/huggingface/accelerate.git@c9fbb71

!pip install -qqq loralib==0.1.1
!pip install -qqq einops==0.6.1

In [ ]:
!pip install -U datasets

In [ ]:
import json
import os
from pprint import pprint
import bitsandbytes as bnb
import torch
import torch.nn as nn
import transformers
from datasets import load_dataset
from huggingface_hub import notebook_login
from peft import (
    LoraConfig,
    PeftConfig,
    PeftModel,
    get_peft_model,
    prepare_model_for_kbit_training
)
from transformers import (
    AutoConfig,
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig
)

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
notebook_login()

In [ ]:
MODEL_NAME = "vilsonrodrigues/falcon-7b-instruct-sharded"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    trust_remote_code=True,
    quantization_config=bnb_config
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
def print_trainable_parameters(model):
  """
  Prints the number of trainable parameters in the model.
  """
  trainable_params = 0
  all_param = 0
  for _, param in model.named_parameters():
    all_param += param.numel()
    if param.requires_grad:
      trainable_params += param.numel()
  print(
      f"trainable params: {trainable_params} || all params: {all_param} || trainables%: {100 * trainable_params / all_param}"
  )

In [ ]:
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)


In [ ]:
config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["query_key_value"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, config)
print_trainable_parameters(model)

In [ ]:
prompt = """
<human>: i feel streesed can you help me?
<assistant>:
""".strip()

In [ ]:
generation_config = model.generation_config
generation_config.max_new_tokens = 200
generation_config.temperature = 0.7
generation_config.top_p = 0.7
generation_config.num_return_sequences = 1
generation_config.pad_token_id = tokenizer.eos_token_id
generation_config.eos_token_id = tokenizer.eos_token_id

In [ ]:
%%time
device = "cuda:0"

encoding = tokenizer(prompt, return_tensors="pt").to(device)
with torch.inference_mode():
  outputs = model.generate(
      input_ids = encoding.input_ids,
      attention_mask = encoding.attention_mask,
      generation_config = generation_config
  )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
data = load_dataset("csv", data_files="/kaggle/input/dataset/train.csv")

In [ ]:
data

In [ ]:
data["train"][0]

In [ ]:
def generate_prompt(data_point):
  return f"""
<human>: {data_point["Context"]}
<assistant>: {data_point["Response"]}
""".strip()

def generate_and_tokenize_prompt(data_point):
  full_prompt = generate_prompt(data_point)
  tokenized_full_prompt = tokenizer(full_prompt, padding=True, truncation=True)
  return tokenized_full_prompt

In [ ]:
data = data["train"].shuffle().map(generate_and_tokenize_prompt)

In [ ]:
data

In [ ]:
from sklearn.model_selection import train_test_split

# Assuming `data` is your dataset
train_data, validation_data = train_test_split(data, test_size=0.2, random_state=42)


In [ ]:
import torch
import math

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    # Ensure logits and labels are tensors
    if isinstance(logits, str):
        raise ValueError("Expected logits to be a tensor, but got string.")
    if isinstance(labels, str):
        raise ValueError("Expected labels to be a tensor, but got string.")
    
    # Reshape logits and labels if necessary (e.g., for sequence length)
    logits = logits.view(-1, logits.size(-1))  # Flatten logits to shape (batch_size * seq_len, vocab_size)
    labels = labels.view(-1)  # Flatten labels for cross entropy loss
    
    # Compute the loss
    loss_fct = torch.nn.CrossEntropyLoss()
    loss = loss_fct(logits, labels)
    
    # Perplexity is the exponentiation of the loss
    perplexity = torch.exp(loss)  # Calculate perplexity
    return {"perplexity": perplexity.item()}

In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset

# Load the Falcon tokenizer
tokenizer = AutoTokenizer.from_pretrained('vilsonrodrigues/falcon-7b-instruct-sharded')

# Ensure pad_token is set
tokenizer.pad_token = tokenizer.eos_token  # Or use a custom pad token

# Example dataset
train_data_dict = {
    'Context': ["Hello, how are you?", "What's your name?"],
    'Response': ["I'm good, thank you!", "My name is AI."]
}

# Convert the dictionary to a Hugging Face Dataset
train_data = Dataset.from_dict(train_data_dict)

# Define the tokenization function
def tokenize_function(examples):
    # Ensure both Context and Response are properly tokenized
    return tokenizer(examples['Context'], examples['Response'], padding="max_length", truncation=True, max_length=512)

# Apply tokenization to the dataset
train_data = train_data.map(tokenize_function, batched=True)

# Check the result
print(train_data)


In [ ]:
# Inspect a single example to verify the format
print(train_data[0])  # Check the first row

In [ ]:
tokenized_input = tokenizer("Hello, how are you?", padding=True, truncation=True, return_tensors="pt")
input_ids = tokenized_input["input_ids"]
attention_mask = tokenized_input["attention_mask"]


In [ ]:
training_args = transformers.TrainingArguments(
      per_device_train_batch_size=1,
      gradient_accumulation_steps=4,
      num_train_epochs=1,
      learning_rate=2e-4,
      fp16=True,
      save_total_limit=3,
      logging_steps=1,
      output_dir="experiments",
      optim="paged_adamw_8bit",
      lr_scheduler_type="cosine",
      warmup_ratio=0.05,
)

trainer = transformers.Trainer(
    model=model,
    train_dataset=data,
    args=training_args,
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False)
)
model.config.use_cache = False
trainer.train()

In [ ]:
model.save_pretrained("trained-model")

In [ ]:
PEFT_MODEL = "****************"

model.push_to_hub(
    PEFT_MODEL, use_auth_token=True
)

In [ ]:
config = PeftConfig.from_pretrained(PEFT_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    config.base_model_name_or_path,
    return_dict=True,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

tokenizer=AutoTokenizer.from_pretrained(config.base_model_name_or_path)
tokenizer.pad_token = tokenizer.eos_token

model = PeftModel.from_pretrained(model, PEFT_MODEL)